# Modelos de Transporte

Os modelos de transporte são amplamente utilizados para apoiar a tomada de decisão em problemas de distribuição logística e alocação de recursos. Sua aplicação é comum em áreas como logística, cadeias de suprimentos, distribuição industrial, planejamento de produção e gestão de estoques.

O problema clássico de transporte é estruturado a partir da existência de fontes de oferta e pontos de demanda, onde cada origem possui determinada capacidade de fornecimento e cada destino apresenta uma necessidade específica. Além disso, considera-se um custo associado ao transporte entre cada origem e destino. A solução do modelo busca encontrar a quantidade ideal a ser transportada entre os pontos, respeitando as **m** restrições de oferta $(0)$ e **n** retrições de demanda $(D)$ e otimizando a função objetivo estabelecida (por exemplo, custo total $Z$).

\begin{equation}
min Z = \sum_{i=1}^{m} \sum_{j=1}^{n} c_{ij}x_{ij}
\end{equation}

sujeito a

\begin{equation}
min Z = \sum_{j=1}^{n} x_{ij} = O_i \qquad i=1, 2, ..., m.
\end{equation}

\begin{equation}
min Z = \sum_{i=1}^{m} x_{ij} = D_j \qquad i=1, 2, ..., n.
\end{equation}

\begin{equation}
x_{ij} \geq 0
\end{equation}


Somando as m restrições de oferta e as n restrições de demanda obtém-se:

\begin{equation}
\sum_{i=1}^{n} O_i = \sum_{j=1}^{n} D_j
\end{equation}

A igualdade indica que o modelo do transporte “exige uma igualdade” entre oferta total e demanda total. Porém o algoritmo a ser apresentado também pode ser utilizado quando a oferta total não for igual a demanda total.  O equilíbrio é essencial para a aplicação dos algoritmos clássicos.

A Tabela a seguir representa um exemplo clássico de um problema de transporte balanceado, no qual três origens devem atender às demandas de três destinos, considerando as quantidades ofertadas e demandadas, bem como as variáveis de decisão associadas ao fluxo de transporte entre cada origem e destino.

| Origens / Destinos | Destino 1 | Destino 2 | Destino 3 | Oferta  |
| ------------------ | :-------: | :-------: | :-------: | :-----: |
| Origem 1           | $x_{11}$  | $x_{12}$  | $x_{13}$  | 40      |
| Origem 2           | $x_{21}$  | $x_{22}$  | $x_{23}$  | 35      |
| Origem 3           | $x_{31}$  | $x_{32}$  | $x_{33}$  | 25      |
| **Demanda**        | **30**    | **45**    | **25**    | **100** |



O diagrama a seguir apresenta a representação gráfica do modelo de transporte, ilustrando as conexões entre as origens e os destinos, bem como as variáveis de decisão associadas ao fluxo de transporte entre cada ponto da rede logística.

```mermaid
graph LR

%% Origens
O1[Origem 1<br>Oferta = 40]
O2[Origem 2<br>Oferta = 35]
O3[Origem 3<br>Oferta = 25]

%% Destinos
D1[Destino 1<br>Demanda = 30]
D2[Destino 2<br>Demanda = 45]
D3[Destino 3<br>Demanda = 25]

%% Ligações da matriz de transporte
O1 -->|x11| D1
O1 -->|x12| D2
O1 -->|x13| D3

O2 -->|x21| D1
O2 -->|x22| D2
O2 -->|x23| D3

O3 -->|x31| D1
O3 -->|x32| D2
O3 -->|x33| D3
```

# Exemplo

Considere os custos agregados a seguir, em reais por tonelada:

| Origens / Destinos | São Paulo        | Recife        | Brasília        | Salvador        | Oferta (t) |
| ------------------ | :--------------: | :-----------: | :-------------: | :-------------: | :--------: |
| Petrolina (PE)     | 180              | 60            | 120             | 80              | 40         |
| Juazeiro (BA)      | 170              | 70            | 110             | 50              | 35         |
| Mossoró (RN)       | 220              | 90            | 150             | 140             | 25         |
| **Demanda (t)**    | **35**           | **25**        | **20**          | **20**          | **100**    |

In [11]:
origens = ["Petrolina", "Juazeiro", "Mossoró"]

destinos = ["São Paulo",
                "Recife",
                "Brasília",
                "Salvador"
]

oferta = {
    "Petrolina": 40,
    "Juazeiro": 35,
    "Mossoró": 25
}

demanda = {
    "São Paulo": 35,
    "Recife": 25,
    "Brasília": 20,
    "Salvador": 20
}

custos = {
("Petrolina","São Paulo"):180,
("Petrolina","Recife"):60,
("Petrolina","Brasília"):120,
("Petrolina","Salvador"):80,

("Juazeiro","São Paulo"):170,
("Juazeiro","Recife"):70,
("Juazeiro","Brasília"):110,
("Juazeiro","Salvador"):50,

("Mossoró","São Paulo"):220,
("Mossoró","Recife"):90,
("Mossoró","Brasília"):150,
("Mossoró","Salvador"):140
}

In [12]:
import pulp as pl
# Definir problema de minimização
modelo = pl.LpProblem("Problema_de_transporte", pl.LpMinimize)

# Decision variables
x = pl.LpVariable.dicts(
    "transporte",
    [(i,j) for i in origens for j in destinos],
    lowBound=0
)

In [13]:
# Definir função objetivo
modelo += pl.lpSum(
    custos[i,j] * x[i,j]
    for i in origens
    for j in destinos
)

In [15]:
# Restrição de oferta
for i in origens:
    modelo += pl.lpSum(
        x[i,j] for j in destinos
    ) == oferta[i]

# Restrição de demanda
for j in destinos:
    modelo += pl.lpSum(
        x[i,j] for i in origens
    ) == demanda[j]

In [17]:
# Solucionar o modelo
modelo.solve()

# Resultados
for i in origens:
    for j in destinos:
        if x[i,j].varValue > 0:
            print(f"{i} → {j}: {x[i,j].varValue}")

print("\nCusto mínimo total:", pl.value(modelo.objective))

Petrolina → São Paulo: 35.0
Petrolina → Recife: 5.0
Juazeiro → Brasília: 15.0
Juazeiro → Salvador: 20.0
Mossoró → Recife: 20.0
Mossoró → Brasília: 5.0

Custo mínimo total: 11800.0
